# Caesar-Chiffre knacken

In der letzten Lektion haben wir die Caesar-Chiffre kennengelernt und in
Python implementiert. Wer den Schlüssel kennt, also die Anzahl Positionen,
um die jeder Buchstabe verschoben wurde, kann einen Chiffretext ohne Mühe
entschlüsseln. Spannender ist die Frage, was passiert, wenn wir den
Schlüssel **nicht** kennen. Genau in dieser Situation befindet sich ein
Angreifer, der eine verschlüsselte Nachricht abfängt.

Das Brechen einer Verschlüsselung ohne Kenntnis des Schlüssels wird als
*Kryptoanalyse* bezeichnet. Für die Caesar-Chiffre gibt es zwei
grundsätzlich verschiedene Vorgehensweisen:

1. **Brute Force**: Alle möglichen Schlüssel werden ausprobiert.
2. **Statistische Analyse**: Aus Eigenschaften des Chiffretextes wird auf
   den Schlüssel geschlossen.

Beide Verfahren setzen wir im Folgenden in Python um.

## Brute Force

Der Begriff *Brute Force* (englisch für «rohe Gewalt») bezeichnet ein
Verfahren, bei dem schlicht alle möglichen Schlüssel der Reihe nach
ausprobiert werden, bis der richtige gefunden ist. Raffinesse ist dabei
nicht gefragt, sondern nur Ausdauer, und davon haben Computer reichlich.

Bei der Caesar-Chiffre ist dieser Ansatz besonders erfolgversprechend,
weil der *Schlüsselraum*, also die Menge aller möglichen Schlüssel,
winzig ist. Das lateinische Alphabet hat 26 Buchstaben. Eine Verschiebung
um 26 Positionen führt wieder zum Klartext, ebenso eine Verschiebung um 0.
Es gibt deshalb nur **25 sinnvolle Schlüssel**. Diese von Hand
durchzuprobieren, dauert ein paar Minuten, ein Computer erledigt es in
Bruchteilen einer Sekunde.

Das Vorgehen ist denkbar einfach:

1. Der Chiffretext wird mit jedem möglichen Schlüssel von 1 bis 25
   entschlüsselt.
2. Die 25 Ergebnisse werden ausgegeben.
3. Unter diesen Kandidaten wird derjenige gesucht, der einen lesbaren
   Text ergibt.

Der letzte Schritt ist der Haken an der Sache: Der Computer liefert zwar
alle Kandidaten, aber er weiss nicht, welcher davon der richtige ist. Bei
einem kurzen Text kann ein Mensch das auf einen Blick erkennen. Soll der
Computer diese Entscheidung selbst treffen, braucht er ein Kriterium, mit
dem er beurteilen kann, wie sehr ein Text nach Deutsch «aussieht». Ein
solches Kriterium liefert die statistische Analyse.

Brute Force zeigt ausserdem, worauf es bei einer sicheren Verschlüsselung
ankommt: Der Schlüsselraum muss so gross sein, dass das Ausprobieren
aller Schlüssel auch mit den schnellsten Computern praktisch unmöglich ist.
Moderne Verfahren wie AES verwenden Schlüssel mit 128 Bit oder mehr, was
rund $3{,}4 \cdot 10^{38}$ möglichen Schlüsseln entspricht.

### Hilfsfunktion

Für beide Verfahren brauchen wir die Funktion `caesar` aus der letzten
Lektion. Wir passen sie leicht an: Zeichen, die keine Buchstaben von `A`
bis `Z` sind (Leerzeichen, Satzzeichen, Umlaute), werden nicht
verschoben, sondern unverändert übernommen. So bleibt ein entschlüsselter
Text besser lesbar.

In [ ]:
def caesar(text: str, shift: int, encrypt=True) -> str:
    text = text.upper()
    if not encrypt:
        shift = -shift
    result = ""

    for char in text:
        if 'A' <= char <= 'Z':
            shifted = (ord(char) - ord('A') + shift) % 26 + ord('A')
            result += chr(shifted)
        else:
            result += char

    return result

Als Beispiel verschlüsseln wir den Anfang des Johannesevangeliums mit
einem Schlüssel, den wir anschliessend «vergessen».

In [ ]:
plain = """Im Anfang war das Wort, und das Wort war bei Gott, und das Wort war
Gott. Dieses war im Anfang bei Gott. Alle Dinge sind durch dasselbe
geworden, und ohne das Wort ist auch nicht eines geworden, das geworden
ist. In ihm war Leben, und das Leben war das Licht für die Menschen."""

ciphertext = caesar(plain, 7)
print(ciphertext)

### Implementation

Die Funktion `brute_force` entschlüsselt den Chiffretext mit jedem der 25
möglichen Schlüssel und gibt die Ergebnisse als Liste von Tupeln
`(schlüssel, kandidat)` zurück. Weil die Kandidaten zurückgegeben und
nicht direkt ausgegeben werden, können wir sie später weiterverarbeiten,
zum Beispiel statistisch bewerten.

In [ ]:
def brute_force(ciphertext: str) -> list[tuple[int, str]]:
    candidates = []

    for shift in range(1, 26):
        candidate = caesar(ciphertext, shift, encrypt=False)
        candidates.append((shift, candidate))

    return candidates

Um die Ausgabe übersichtlich zu halten, zeigen wir von jedem Kandidaten
nur die ersten 40 Zeichen an.

In [ ]:
for shift, candidate in brute_force(ciphertext):
    print(f"{shift:2}: {candidate[:40]}")

Ein Blick auf die Liste genügt, um den richtigen Schlüssel zu finden.
Der Computer kann das aber noch nicht. Dazu brauchen wir die statistische
Analyse.

## Statistische Analyse

Die statistische Analyse, genauer die *Häufigkeitsanalyse*, nutzt eine
Schwäche der Caesar-Chiffre aus: Jeder Buchstabe des Klartextes wird
immer durch denselben Buchstaben im Chiffretext ersetzt. Die Buchstaben
werden also zwar umbenannt, aber ihre Häufigkeit bleibt erhalten.

In jeder natürlichen Sprache kommen die Buchstaben unterschiedlich oft
vor. Im Deutschen ist `E` mit rund 17 % mit Abstand der häufigste
Buchstabe, gefolgt von `N` (knapp 10 %) und `I` (gut 7.5 %). Buchstaben
wie `Q`, `X` oder `Y` kommen dagegen kaum vor. Diese Verteilung ist für
eine Sprache so charakteristisch, dass sie sich in jedem genügend langen
Text wiederfindet.

Daraus ergibt sich folgendes Vorgehen:

1. Im Chiffretext wird gezählt, wie oft jeder Buchstabe vorkommt.
2. Der häufigste Buchstabe im Chiffretext entspricht sehr wahrscheinlich
   dem `E` im Klartext.
3. Der Abstand zwischen diesem Buchstaben und `E` im Alphabet ist der
   gesuchte Schlüssel.

Ist im Chiffretext beispielsweise `I` der häufigste Buchstabe, liegt eine
Verschiebung um vier Positionen nahe, denn `E` + 4 = `I`.

Dieses Verfahren hat seine Grenzen. Bei sehr kurzen Texten weicht die
Häufigkeitsverteilung oft deutlich vom Durchschnitt ab, und der häufigste
Buchstabe ist dann nicht unbedingt ein `E`. Zuverlässiger wird die Analyse,
wenn nicht nur ein einzelner Buchstabe, sondern die gesamte Verteilung
betrachtet wird: Für jeden der 25 möglichen Schlüssel wird geprüft, wie
gut die Buchstabenhäufigkeiten des entschlüsselten Textes mit den
bekannten Häufigkeiten der deutschen Sprache übereinstimmen. Der Schlüssel
mit der besten Übereinstimmung gewinnt.

Damit schliesst sich der Kreis zu Brute Force: Die Kombination beider
Ansätze, also alle Schlüssel ausprobieren und das Ergebnis statistisch
bewerten, erlaubt es dem Computer, eine Caesar-Chiffre vollständig
selbständig zu knacken.

Die Häufigkeitsanalyse ist übrigens keine moderne Erfindung. Sie wurde
bereits im 9. Jahrhundert vom arabischen Gelehrten al-Kindi beschrieben
und funktioniert nicht nur bei der Caesar-Chiffre, sondern bei allen
einfachen Substitutionschiffren.

### Buchstabenhäufigkeit zählen

Als erstes brauchen wir die bekannten Häufigkeiten der deutschen Sprache.
Wir übernehmen die Werte (in Prozent) aus der Häufigkeitstabelle der
letzten Lektion in ein Dictionary.

In [ ]:
GERMAN_FREQUENCY = {
    'A': 6.51, 'B': 1.89, 'C': 3.06, 'D': 5.08, 'E': 17.40, 'F': 1.66,
    'G': 3.01, 'H': 4.76, 'I': 7.55, 'J': 0.27, 'K': 1.21, 'L': 3.44,
    'M': 2.53, 'N': 9.78, 'O': 2.51, 'P': 0.79, 'Q': 0.02, 'R': 7.00,
    'S': 7.27, 'T': 6.15, 'U': 4.35, 'V': 0.67, 'W': 1.89, 'X': 0.03,
    'Y': 0.04, 'Z': 1.13,
}

Die Funktion `letter_frequency` zählt, wie oft jeder Buchstabe in einem
Text vorkommt, und rechnet die Anzahl in Prozent um. Im Unterschied zur
letzten Lektion werden alle anderen Zeichen ignoriert, und das Dictionary
enthält jeden Buchstaben von `A` bis `Z`, auch wenn er gar nicht
vorkommt. Das brauchen wir später für den Vergleich mit
`GERMAN_FREQUENCY`.

In [ ]:
def letter_frequency(text: str) -> dict[str, float]:
    counts = {}
    for code in range(ord('A'), ord('Z') + 1):
        counts[chr(code)] = 0

    total_letters = 0
    for char in text.upper():
        if char in counts:
            counts[char] += 1
            total_letters += 1

    frequency = {}
    for letter, count in counts.items():
        if total_letters > 0:
            frequency[letter] = count / total_letters * 100
        else:
            frequency[letter] = 0.0

    return frequency

In [ ]:
frequency = letter_frequency(ciphertext)

for letter, percent in sorted(frequency.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f"{letter}: {percent:5.2f} %")

### Der häufigste Buchstabe

Der häufigste Buchstabe im Chiffretext entspricht vermutlich dem `E`. Die
Funktion `find_shift` aus der letzten Lektion berechnet den Abstand
zwischen zwei Buchstaben. Die Funktion `crack_by_e` sucht den häufigsten
Buchstaben und gibt den daraus berechneten Schlüssel zurück.

In [ ]:
def find_shift(char: str, e='E') -> int:
    return (ord(char) - ord(e)) % 26


def crack_by_e(ciphertext: str) -> int:
    frequency = letter_frequency(ciphertext)
    most_common = max(frequency, key=frequency.get)
    return find_shift(most_common)

In [ ]:
shift = crack_by_e(ciphertext)
print(f"Vermuteter Schlüssel: {shift}")
print(caesar(ciphertext, shift, encrypt=False))

Bei einem kurzen Text geht das schief. Im folgenden Beispiel ist `S` und
nicht `E` der häufigste Buchstabe des Klartextes.

In [ ]:
short_cipher = caesar("Alle Truppen sammeln sich am Fluss", 3)
shift = crack_by_e(short_cipher)

print(f"Vermuteter Schlüssel: {shift}")
print(caesar(short_cipher, shift, encrypt=False))

::: {.callout-note icon=false}
## Variante: Die gesamte Verteilung vergleichen

Zuverlässiger ist es, die gesamte Häufigkeitsverteilung zu betrachten. Um
zu messen, wie gut die beobachteten Häufigkeiten $B_i$ eines Textes zu
den erwarteten Häufigkeiten $E_i$ der deutschen Sprache passen, verwenden
wir die *Chi-Quadrat-Summe*:

$$
\chi^2 = \sum_{i=A}^{Z} \frac{(B_i - E_i)^2}{E_i}
$$

Für jeden Buchstaben wird die Abweichung zwischen beobachtetem und
erwartetem Wert quadriert und durch den erwarteten Wert geteilt. Das
Quadrieren sorgt dafür, dass sich positive und negative Abweichungen
nicht gegenseitig aufheben. Das Teilen durch $E_i$ bewirkt, dass eine
Abweichung bei einem seltenen Buchstaben stärker ins Gewicht fällt: Ein
Text mit 5 % `Q` ist viel verdächtiger als einer mit 5 % zu wenig `E`.

Je **kleiner** $\chi^2$, desto ähnlicher ist der Text einem deutschen
Text.

In [ ]:
def chi_squared(text: str) -> float:
    observed = letter_frequency(text)
    total = 0.0

    for letter, expected in GERMAN_FREQUENCY.items():
        total += (observed[letter] - expected) ** 2 / expected

    return total

Nun verbinden wir die beiden Verfahren: `crack` erzeugt mit
`brute_force` alle Kandidaten, bewertet jeden mit `chi_squared` und gibt
den Kandidaten mit dem kleinsten Wert zurück.

In [ ]:
def crack(ciphertext: str) -> tuple[int, str]:
    best_shift = 0
    best_text = ciphertext
    best_score = float('inf')

    for shift, candidate in brute_force(ciphertext):
        score = chi_squared(candidate)
        if score < best_score:
            best_shift = shift
            best_text = candidate
            best_score = score

    return best_shift, best_text

In [ ]:
shift, text = crack(ciphertext)
print(f"Gefundener Schlüssel: {shift}")
print(text)

Und auch der kurze Text, an dem die erste Variante gescheitert ist, wird
nun richtig entschlüsselt.

In [ ]:
shift, text = crack(short_cipher)
print(f"Gefundener Schlüssel: {shift}")
print(text)

Ganz ohne Grenzen ist aber auch dieses Verfahren nicht. Bei einem Text
wie «Hallo Welt, alles klar?» mit nur 18 Buchstaben, davon sechs `L`, ist
die Verteilung so untypisch, dass auch die Chi-Quadrat-Summe den falschen
Schlüssel liefert. Je länger der Chiffretext, desto zuverlässiger
funktioniert die statistische Analyse.

Wer nachvollziehen möchte, wie die Entscheidung zustande kommt, kann sich
die Werte für alle Schlüssel ausgeben lassen. Der richtige Schlüssel
sticht deutlich hervor.

In [ ]:
for shift, candidate in brute_force(ciphertext):
    print(f"{shift:2}: {chi_squared(candidate):10.1f}  {candidate[:30]}")

:::